# Main file 

05.11.2025

Written by: Arnav Kumar Upadhyay, Cedric Meyer adapted from MATLAB code of Marc, Jan Spindler DOI-https://doi.org/10.1021/acssynbio.4c00578

This script is the main script for the TXTL v2 model in INDEPENDENT GENE SIMULATION mode.

The script simulates each gene independently without competition for shared resources.


In [2]:
# Packages functions and data types --------
include("packages.jl") 

# Sequences 
sequence_file	    = "Ecoli_K12_MG1655_geneset_cds_hpaBC_ddc.fa"    # "lacS_background.fa"
hybrid_file		    = "lacS.txt"
seqs_to_use 	    = (1,2323)                                 # Range of sequences in Fasta file. Takes all sequences from sequence a to sequence b
num_background_genes = 0
heterologous_genes  = ["hpaBC"]# ,"ddc"]

# initialization of data containers
scale = txtl_types.init_scale()         # struct that makes it possible to influence rates by scaling them

µ = 1                    # A growth rate (0.1 to 1.8) can be chosen and the proteins of the TXTL machinery will be tuned according to that growth rate and the processivity of the RNA polymerase

## flags for efficiency factors
positive_aminoacid	= true 
anti_shinedalgarno	= false      # If no hybridization file is available put flag to false. We apply the intaRNA program's intarna function (q: aSD, t: sequence, SeedBP: 5, out: tMinE) to generate a hybridization profile along the sequence. Link to the website: https://rna.informatik.uni-freiburg.de/IntaRNA/Input.jsp;jsessionid=2AE257E91E7276283F22CD77A5A8AA03
dynamic_efficiency	= true 
save_data           = false

flags = txtl_types.flags(positive_aminoacid, anti_shinedalgarno, dynamic_efficiency)

# Data import --------
mRNA 		= to_DataFrame(readmatrix("../data/input/mRNA_concentrations.xlsx", "mRNA_concentration_high_cell_v", "A1:B2324"))
mRNA.mRNA 	= Float64.(mRNA.mRNA);

background_genes = String.(first(mRNA.gene, num_background_genes))
genes = vcat(background_genes, heterologous_genes)
println("Number of sequences: $(length(genes))") 

mRNA = subset(mRNA, :gene => ByRow(in(genes)))


## Import information about model sp ecies and sequences 
seqs, seq_names, hyb_file_dict, species_and_seqs, idx = species_and_sequences(genes, µ, sequence_file, hybrid_file, flags, seqs_to_use)
# Are mRNA gene names in E_coli gene 


## Calculate efficiency factors 
efac_pos_aa_dict, efac_int_sd_dict, dynamic_efficiencies = efficiency_factor_initialization(seqs, seq_names, hyb_file_dict, species_and_seqs)

## calculation of initial concentrations
y0, initial_concentrations = calculation_of_initial_concentration(species_and_seqs, seqs, seq_names, scale, idx);

## calculate and load kinetic parameters
p_met, p_seq, p_dyn = parameters(initial_concentrations, species_and_seqs);

## Release factor balance
RF_balance!(y0, initial_concentrations)

# Preintegration
## Definition of mass matrix
conserved = [idx.EFG, idx.EFD] # EFG, EFTu_GDP - Conserved species, Algebraic equations are given by mass balances

M = Matrix{Float64}(I, length(y0), length(y0)) # Creates unity matrix in the size of y0

M[conserved, :] .= 0; # setting mass matrix to zero to  remove time derivatives from those variables. This converts these differential equations into algebraic constraints


tags_pre = txtl_types.data_struct_preintegration(scale, idx, species_and_seqs, flags, dynamic_efficiencies, initial_concentrations, p_met, p_seq, p_dyn, mRNA.mRNA[1])

## Integration of the DAE system without DNA in the system 
y_steady_state = preintegration(tags_pre, y0, M);


# Elongation factor balance 
EF_balance!(tags_pre, y0, y_steady_state)


# Build states vector for multiple sequences 
y0_all = combined_states_vector(tags_pre, y_steady_state, seqs, seq_names)

# include_mRNA_concentrations!(y0_all, tags_pre, seq_names, seqs, mRNA)

# Prepare per-sequence tags (for use in ODE)
local_tags_list = generate_local_tags(seqs, seq_names, tags_pre, efac_pos_aa_dict, efac_int_sd_dict, mRNA);

# Integration
M = Matrix{Float64}(I, length(y0_all), length(y0_all))

M[conserved, :] .= 0

tspan 	= (0.0, 100.0) ## Define time span for the simulation

p_parallel = local_tags_list

f 		= ODEFunction(ODE_parallel_distributed!, mass_matrix = M) ## Define ODEFunction to make use of mass matrix

prob 	= ODEProblem(f, y0_all, tspan, p_parallel); ## Create an ODE problem

sol = solve(prob, FBDF(autodiff=false), reltol = 1e-8, abstol = 1e-6); ## Solve ODE system


🔧 Julia started with 8 threads
Number of sequences: 1


In [3]:
# Inputs to test functions
du = similar(y0_all)
du = fill!(du, 0.0);
u = sol[end];
t = tspan;

# function ODE_parallel_distributed!(du, u, p, t)

    local_tags_list = p_parallel				# Extraction of local_tags_list from p
    tags            = local_tags_list[1]
    n_metab = length(tags.species_and_sequences.Tag) -4 		# number of metabolites in the shared pool that is used by all sequences 
    @views y_metab =  u[1:n_metab]				# Save all pool species from u in a vector 
    idx = n_metab						# Index of last pool species


	y_codons_lengths = Int[length(local_tags_list[i].species_and_sequences.Seq) + 3 for i in eachindex(local_tags_list)]	# List with sequence lengths - 1 stop codon + 5 observable variables


	y_codons = Vector{SubArray{Float64,1,typeof(u),Tuple{UnitRange{Int64}},true}}(undef, length(y_codons_lengths))
    idx = n_metab
    @views for (i,L) in enumerate(y_codons_lengths)
        y_codons[i] = u[idx+1:idx+L]  # ✅ View statt Kopie
        idx += L
    end

    
    n_seq = tags.species_and_sequences.no_of_seqs 		# save number of sequences
    du_metab_all = [zeros(n_metab) for _ in 1:n_seq]	# List to store a du vector for the metabolite pool for every sequence 
    du_codons = Vector{Vector{Float64}}(undef, n_seq)	# List to store a du vector for the codons for every sequence 

    # THREADED parallelization of per-sequence rate calculations
    # ONLY CHANGE: map -> Threads.@threads for
    results = Vector{Vector{Float64}}(undef, n_seq)  # Pre-allocate results array

    i = 1
 
    local_tags = local_tags_list[i]		# Extraction of local tags
    u_seq = vcat(y_metab, y_codons[i])				# Sequenzspezifischer state vector
    du_seq = zeros(length(u_seq))					# Sequenzspezifischer rates vector

    fill!(du, 0.0)  # Reset to zero, just like initializing with zeros


    rhs_metabolites!(du, u, local_tags)
    for i in du println(i) end

    # @benchmark rhs_sequence!(du, u, local_tags)
    rhs_sequence!(du, u, local_tags)

    


-0.0008558745948903379
-0.0008558735078555684
-2.1781674771140236e-12
-1.4866360237026505e-10
-1.0870347609244838e-9
1.0717258192016743e-9
-1.33383082356886e-10
1.3091749906379846e-11
3.4145186678102846e-11
1.0142672986213075e-10
0.0
-3.410605131648481e-13
0.6170702716917731
-0.6170702716917731
4.611135473453487e-9
0.0
0.0
0.0
-1.0142672129693359e-10
-0.03937017059269363
-0.017973343596677273
-0.011126354969165975
-0.03765843477978209
-0.03337905735138204
-0.021396837763293982
-0.005135241688982646
-0.0368025601355911
-0.017973283643802605
-0.03166731753574782
-0.02482033213780069
-0.030889275634247013
-0.010270476579281257
-0.005913284591654323
-0.0016882689044767384
-0.004302845479757129
-0.02396445533999185
-0.02909969869424541
-0.008380875301759107
-0.0009626822084472039
-0.013765026417341994
-0.008558734991264857
-0.016261596041475302
-0.03423494097434865
-3.51764054946144e-19
-4.418665855054284e-19
-2.7729590338634534e-19
-0.012271995134168552
-0.001428386347665299
-0.01454984971

835-element Vector{Float64}:
 -1.645878204407404e-8
 -1.537174727458436e-8
 -2.1781674771140236e-12
 -1.4866360237026505e-10
 -1.0870347609244838e-9
  1.0717258192016743e-9
 -1.33383082356886e-10
  1.3091749906379846e-11
  3.4145186678102846e-11
  1.0142672986213075e-10
  0.0
 -3.410605131648481e-13
 -2.1696469320930944e-5
  ⋮
  2.4230518002308667e-8
  3.9421389871202583e-8
  3.9792170345885267e-8
  2.998395599732376e-8
  2.9384965262923765e-8
  4.158957620946282e-8
  5.6983092306902607e-8
  6.908975576326234e-8
  7.004549962155802e-8
  7.090338543873885e-8
  7.166107099105916e-8
  1.2696358337741182e-6

In [12]:
# du = zeros(Float64, length(u))
tags = local_tags
# Unpack parameters 
@unpack µ, k_proc, k_cat_Pol, C_RNP, V_TCmax, K_M_D, k_cat_E, C_RNE, V_degmax, K_M_deg, k_D, n_DNA_t, n_DNA_s = tags.p_seq.TX
@unpack k_TLEmaxj, KM_T3j, KM_EFGGTP = tags.p_seq.EL
@unpack k_TE, KM_RK, KM_GTP, KM_RF3GTP, k_RF3_diss_GDP, k_RF1_diss_GTP, k_GTP_hydr, k_RF1_ass = tags.p_seq.TER
@unpack k_RF3_GDP_ass, k_RF3_GTP_ass, k_RF3_GDP_diss, k_RF3_GTP_diss = tags.p_seq.RF3
@unpack A1, A2, A3, A4, A5, k_cat_K_M_G1, k_cat_K_M_G2, k_RRF, q_RRF, one_q_G2_k_split = tags.p_seq.PTC 

# Unpack constant variables 
ATP, GTP, UTP, CTP, ADP, GDP, UDP, CDP, AMP, GMP, UMP, CMP, Eut, Est, RFt, RRF, Ala, Cys, 
Asp, Glu, Phe, Gly, His, Ile, Lys, Leu, Met, Asn, Pro, Gln, Arg, Ser, Thr, val, Trp, Tyr, C_D = tags.init_c.constant_vars 

# Unpack dynamic variables 
S30, S50, IF1, IF2, IF3, SC1, SC2, SC3, SC4, PIC, EFG, EFD, EGT, EGD, EFT, RF3, RGT, RGD, b1t, ta1, ta2, tc1, td1, te2, tf1, tg2, tg3, th1, ti1, tk1, t1l, t2l, t3l, t4l, t5l, tm1, tn1, tp1, tp2, tp3, 
tq1, tq2, tr2, tr3, tr4, tr5, ts1, ts2, ts3, ts5, tt1, tt2, tt4, tv1, tv2, tw1, ty1, tA1, tA2, tC1, tD1, tE2, tF1, tG2, tG3, tH1, tI1, tK1, tL1, tL2, tL3, tL4, tL5, tM1, tN1, tP1, tP2, tP3, tQ1, tQ2, 
tR2, tR3, tR4, tR5, tS1, tS2, tS3, tS5, tT1, tT2, tT4, tV1, tV2, tW1, tY1, a1t, a2t, c1t, d1t, e2t, f1t, g2t, g3t, h1t, i1t, k1t, l1t, l2t, l3t, l4t, l5t, m1t, n1t, p1t, p2t, p3t, q1t, q2t, r2t, r3t, 
r4t, r5t, s1t, s2t, s3t, s5t, t1t, t2t, t4t, v1t, v2t, w1t, y1t, S70, rna, PRT, PTC = u 

RNA = tags.RNA

# Containers 
num_species = length(tags.species_and_sequences.Name)
len_sequence = length(tags.species_and_sequences.Seq) # Codon length with stop codon

# Define RIB1

rate_RIB        = zeros(Float64, len_sequence-1);
@views c_RIB	= u[tags.idx_first_codon:end]

 

@views total_RNA   = RNA + sum(c_RIB[1:12])

q_R_elo     = ones(len_sequence-1);
    
@views for i in 2:(len_sequence - 13)

    if total_RNA-sum(c_RIB[i:i+12]) == 0

        q_R_elo[i] 	= 0;
    else
        q_R_elo[i] 	= 1 - c_RIB[i+12] / (total_RNA-sum(c_RIB[i:i+11]));
    
    end
end 


@views for i in (len_sequence-13) : (len_sequence-1)
    if total_RNA - sum(c_RIB[i : end]) == 0
        q_R_elo[i] = 0
    else 
        q_R_elo[i] = 1 - (c_RIB[end] + PTC) / (total_RNA - sum(c_RIB[i:end-1]))
    end
end

# dynamic efficiency factor 
if tags.flags.dynamic_efficiency == true

    efac = dynEffficiencyCalculation(tags, EGT, u);

end
    

# V_EL
C_vect = reshape(u[1:num_species], 1, num_species)  # Directly assign
c_Codon = tags.t3_Codon_struc * C_vect'

V_TL_EL	= zeros(len_sequence-1);

EGT_inv = inv(EGT)


@inbounds for i in 1:len_sequence-2

    cod_next = tags.species_and_sequences.Seq[i+1]
    cod_curr = tags.species_and_sequences.Seq[i]

    j = tags.species_and_sequences.cod_idx[cod_next]    # O(1) Lookup
    k = tags.species_and_sequences.cod_idx[cod_curr]    # O(1) Lookup

    cc = c_Codon[j]

    if cc == 0
        V_TL_EL[i+1] = 0
    else
        eff = efac[k] * min(tags.efac_pos_aa[i], tags.efac_int_sd[i])

        denom = 1 + KM_T3j/cc + KM_EFGGTP*EGT_inv

        V_TL_EL[i+1] =
            q_R_elo[i] * k_TLEmaxj * c_RIB[i] * eff / denom
    end
end

V_TL_EL_sum = sum(V_TL_EL);

c_codon_seq = tags.matrix_N * c_Codon

N = C_vect[1,:] * (1 ./c_codon_seq)'

133×698 Matrix{Float64}:
  9.63408     3.89182     13.5856      …  10.8891      10.8891
  9.64319     3.8955      13.5984         10.8994      10.8994
  0.00564896  0.00228197   0.00796593      0.00638483   0.00638483
  0.0100555   0.00406206   0.0141798       0.0113654    0.0113654
  1.14704     0.463362     1.61751         1.29646      1.29646
  0.741621    0.299588     1.0458      …   0.838229     0.838229
  0.318679    0.128735     0.449388        0.360192     0.360192
  0.358363    0.144765     0.505348        0.405045     0.405045
  0.101125    0.0408509    0.142602        0.114298     0.114298
  1.13403     0.458109     1.59917         1.28176      1.28176
  0.0454644   0.018366     0.064112    …   0.0513869    0.0513869
 11.5974      4.68492     16.3541         13.1081      13.1081
  7.58348     3.06345     10.6939          8.57135      8.57135
  ⋮                                    ⋱               
  0.580326    0.234431     0.818352        0.655923     0.655923
  1.9529      

In [13]:
C_vect[1,:]

133-element Vector{Float64}:
 20.065442041779477
 20.084414952453248
  0.01176540222947059
  0.02094314555087542
  2.3889999533522275
  1.54461500704208
  0.6637306843811214
  0.7463818107365565
  0.2106189983080497
  2.36191750016045
  0.0946911944619717
 24.154502728924673
 15.794527928383365
  ⋮
  1.2086774476935969
  4.067422507065785
  0.814805991967685
  3.261355349041788
  2.010964573924635
  3.156959716930158
  1.5316394747878637
  2.633098982720315
  9.628039556867991
  3.5349116346386875
  2.3775990747974443
  5.007437008269146

In [14]:
c_codon_seq

698×1 Matrix{Float64}:
  2.0827554026756108
  5.155799754059062
  1.4769660307652477
 12.857423219504994
  6.532548826597779
  2.9223356082406715
 12.169230627006367
  1.7325027635621744
  3.261355349041788
  3.156959716930158
  2.215012548790431
 12.169230627006367
  3.375131448599875
  ⋮
  9.60060460236495
  9.60060460236495
 10.048761183137568
  9.60060460236495
  3.245135411572915
 12.857423219504994
  1.842712206403487
  1.842712206403487
  1.842712206403487
  1.842712206403487
  1.842712206403487
  1.842712206403487

In [18]:
C_vect[1,:]

133-element Vector{Float64}:
 20.065442041779477
 20.084414952453248
  0.01176540222947059
  0.02094314555087542
  2.3889999533522275
  1.54461500704208
  0.6637306843811214
  0.7463818107365565
  0.2106189983080497
  2.36191750016045
  0.0946911944619717
 24.154502728924673
 15.794527928383365
  ⋮
  1.2086774476935969
  4.067422507065785
  0.814805991967685
  3.261355349041788
  2.010964573924635
  3.156959716930158
  1.5316394747878637
  2.633098982720315
  9.628039556867991
  3.5349116346386875
  2.3775990747974443
  5.007437008269146

In [16]:
N = C_vect[1,:] * (1 ./c_codon_seq)'

133×698 Matrix{Float64}:
  9.63408     3.89182     13.5856      …  10.8891      10.8891
  9.64319     3.8955      13.5984         10.8994      10.8994
  0.00564896  0.00228197   0.00796593      0.00638483   0.00638483
  0.0100555   0.00406206   0.0141798       0.0113654    0.0113654
  1.14704     0.463362     1.61751         1.29646      1.29646
  0.741621    0.299588     1.0458      …   0.838229     0.838229
  0.318679    0.128735     0.449388        0.360192     0.360192
  0.358363    0.144765     0.505348        0.405045     0.405045
  0.101125    0.0408509    0.142602        0.114298     0.114298
  1.13403     0.458109     1.59917         1.28176      1.28176
  0.0454644   0.018366     0.064112    …   0.0513869    0.0513869
 11.5974      4.68492     16.3541         13.1081      13.1081
  7.58348     3.06345     10.6939          8.57135      8.57135
  ⋮                                    ⋱               
  0.580326    0.234431     0.818352        0.655923     0.655923
  1.9529      

In [19]:
(1 ./c_codon_seq)'

1×698 adjoint(::Matrix{Float64}) with eltype Float64:
 0.480133  0.193956  0.677064  0.0777761  …  0.542678  0.542678  0.542678